**SOC612: Data Analytics for the Social Sciences**

**Date: September 29, 2026**

**Author: R. Duerr**

**Lecture 4: Inferential analysis**

To apply some of the most used inferential regression techniques in sociology, we use a pre-processed dataset of the World Value Survey Wave 6-7 (2022).

At the beginning, we load the edited dataset, and do some minor recodings. The content for lecture 4 starts at chunk 1.

We have the following 10 variables next to the case ID in our dataset called 'data':

- cntry: the three-letter country code
- sex, age, edu: demographic information on the respondents sex, age, and education level.
- chldrn: the number of children the respondents have.
- relpers: indicates as what type of religious person the respondents see themselves.
- relm1, relm2: two variables showing opposing meanings of religion; first, whether the focus of religion should be on doing good for others, or on norms and ceremonies. Second, if religion lessons should focus on the present life or the afterlife.
- sci_sum, rel_sum: two cumulative index variables showing how positive the respondents are towards science and technology, and towards religion.

In [ ]:
import pandas as pd

# Read the data
data = pd.read_csv("wb_67reg.csv")

# Convert columns 2, 3, 5, 7, 8, 9 (0-indexed: 1, 2, 4, 6, 7, 8) to categorical
cols = [1, 2, 4, 6, 7, 8]
data[data.columns[cols]] = data[data.columns[cols]].apply(lambda x: x.astype("category"))

# Relevel edu (sets order, first level = reference)
data["edu"] = pd.Categorical(
    data["edu"],
    categories=["primary_or_below", "secondary", "post_secondary", "tertiary"]
)

# Relevel relpers
data["relpers"] = pd.Categorical(
    data["relpers"],
    categories=["religious", "not_religious", "atheist"]
)

# Filter for Hong Kong
data_HK = data[data["cntry"] == "HKG"].copy()

For our first model on the Hong Kong subset, we apply a simple linear regression (Model 1). Our dependent variable is a cumulative index of a person's religiosity (rel_sum). Our sole predictor here is a cumulative index of a person's attitude towards science and technology. Both are positively coded, so the higher each of them is, the stronger their attitude towards either.

We extend the number of predictors in Model 2: we add 'sex', 'age', and 'edu' as additional predictors to the model. For both models, we make a prediction based on our coefficients.

As for now, we hold the model assumptions to be true. We will cover assumptions, model improvements, robustness, and selection in a later session.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

##### [MODEL 1:] Simple linear model
model1 = smf.ols("rel_sum ~ sci_sum", data=data_HK).fit()
print(model1.summary())

# graph
plt.figure(figsize=(8, 6))
sns.regplot(
    data=data_HK, x="sci_sum", y="rel_sum",
    scatter_kws={"alpha": 0.6},        # jitter effect via transparency
    ci=95,                             # confidence interval band (se=T)
    scatter=True
)
plt.title("Association between Science and Religion")
plt.xlabel("Science Index")
plt.ylabel("Religion Index")
plt.figtext(0.5, -0.02, "Data: WVS 6-7, Hong Kong Subset (n=1,918)",
            ha="center", fontsize=9)
plt.tight_layout()
plt.show()

##### [MODEL 2:] Multiple linear model
model2 = smf.ols("rel_sum ~ sci_sum + sex + age + edu", data=data_HK).fit()
print(model2.summary())

# Model 1: Predictions for sci_sum values 0, 10, 20, 30, 40, 50, 60, and 100
new1 = pd.DataFrame({"sci_sum": [0, 10, 20, 30, 40, 50, 60, 100]})
pred1 = model1.get_prediction(new1).summary_frame(alpha=0.05)
print(pred1)  # columns: mean, mean_ci_lower, mean_ci_upper (like "fit/lwr/upr")

# Model 2: Prediction for mean sci_sum, female, mean age, and post_secondary
new2 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()],
    "sex": ["female"],
    "age": [data_HK["age"].mean()],
    "edu": ["post_secondary"]
})
pred2 = model2.get_prediction(new2).summary_frame(alpha=0.05)
print(pred2)

There are cases when we want to modify our variables, possibly because the regression line does not show a good fit, or we have a specific aim for communicating the results. The regression and prediction function works in the same way, although interpretation changes.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

##### [MODEL 3:] Transformed Variable Models

# R's scale() = z-standardization. Create standardized columns explicitly
# (patsy formulas don't have a scale() function):
data_HK["rel_sum_z"] = (data_HK["rel_sum"] - data_HK["rel_sum"].mean()) / data_HK["rel_sum"].std(ddof=1)
data_HK["sci_sum_z"] = (data_HK["sci_sum"] - data_HK["sci_sum"].mean()) / data_HK["sci_sum"].std(ddof=1)

model3 = smf.ols("rel_sum_z ~ sci_sum_z + sex + age + I(age ** 2) + edu", data=data_HK).fit()
print(model3.summary())

# Model 3: Prediction for mean sci_sum (= 0 when scaled), male, mean age, post_secondary
new3 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()],
    "sex": ["male"],
    "age": [data_HK["age"].mean()],
    "edu": ["post_secondary"]
})
# Standardize the new sci_sum with the SAME mean/sd used for the model
new3["sci_sum_z"] = (new3["sci_sum"] - data_HK["sci_sum"].mean()) / data_HK["sci_sum"].std(ddof=1)
pred3 = model3.get_prediction(new3).summary_frame(alpha=0.05)
print(pred3)

In sociology, we are often dealing with categorical outcomes, not continuous ones, and our coefficients show changes in odds ratios and not absolute values. We therefore often rely on logistic regression models. We can estimate logistic regression models for both binary and multi-category outcomes. In the following chunk, we estimate first a linear probability model (Model 4), and its obvious shortcoming. Model 5 shows a conventional binary logistic regression model, with religious meaning as the dependent variable and the religion and science index, and sex, age, and education as other covariates.

In Model 6 (ordered logistic regression) and Model 7 (multinomial logistic regression), we look at multiple outcomes.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
import matplotlib.pyplot as plt
import seaborn as sns

##### [MODEL 4:] Linear probability model
model4 = smf.ols("relm2_num ~ rel_sum", data=data_HK).fit()
print(model4.summary())

# graph (relm2 needs to be numeric for plotting — create it first, see below)
plt.figure(figsize=(8, 6))
sns.regplot(data=data_HK, x="rel_sum", y="relm2_num",
            scatter_kws={"alpha": 0.6}, ci=None)  # ci=None → se=F
plt.title("Association between Science and Religion")
plt.xlabel("Religion Index")
plt.ylabel("Religious Meaning")
plt.figtext(0.5, -0.02, "Data: WVS 6-7, Hong Kong Subset (n=1,918)",
            ha="center", fontsize=9)
plt.tight_layout()
plt.show()

##### [MODEL 5:] Binary logistic regression model
model5 = smf.logit("relm2 ~ rel_sum + sci_sum + sex + age + edu",
                   data=data_HK).fit()
print(model5.summary())

# prediction (type="response" → probabilities)
new5 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()],
    "rel_sum": [data_HK["rel_sum"].mean()],
    "sex": ["male"],
    "age": [data_HK["age"].mean()],
    "edu": ["tertiary"]
})
pred5 = model5.predict(new5)
print(pred5)  # predicted probability

##### [MODEL 6:] Ordered logistic regression
# statsmodels' OrderedModel needs a categorical (or integer-coded) endog
model6 = OrderedModel(
    data_HK["relpers"],
    data_HK[["rel_sum", "sci_sum", "sex", "age", "edu"]],
    distr="logit"
).fit(method="bfgs")
print(model6.summary())

# Model 6: Prediction — predicted probabilities per class
new6 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()],
    "rel_sum": [data_HK["rel_sum"].mean()],
    "sex": ["female"],
    "age": [data_HK["age"].mean()],
    "edu": ["tertiary"]
})
pred6_probs = model6.predict(new6)          # probabilities for each level
print(pred6_probs)
# type="class" → pick the level with the highest probability:
pred6_class = pred6_probs.columns[np.argmax(pred6_probs.values)]
print("Predicted class:", pred6_class)

##### [MODEL 7:] Multinomial logistic regression
# Relevel so "religious" is the reference (baseline) category, as in your model6/7 setup
data_HK["relpers_mn"] = pd.Categorical(
    data_HK["relpers"],
    categories=["religious", "not_religious", "atheist"]
)

model7 = smf.mnlogit("relpers_mn ~ rel_sum + sci_sum + sex + age + edu",
                     data=data_HK).fit()
print(model7.summary())

# Model 7: Prediction for male and female
new7 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()] * 2,
    "rel_sum": [data_HK["rel_sum"].mean()] * 2,
    "sex": ["male", "female"],
    "age": [data_HK["age"].mean()] * 2,
    "edu": ["tertiary"] * 2
})
pred7 = model7.predict(new7)  # probabilities per category (rows = newdata rows)
print(pred7)

It's also common in sociology to work with count data. They show the number of an event. The Poisson regression and the Negative Binomial Regression are the most typical ones. For models 8 and 9, we estimate the number of children someone has, based on their religion and science indices, sex, age, and education.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

##### [MODEL 8:] Poisson regression
model8 = smf.poisson("chldrn ~ rel_sum + sci_sum + sex + age + edu",
                    data=data_HK).fit()
print(model8.summary())

# Model 8: Prediction — mean sci_sum, min & max rel_sum, female, mean age, tertiary
new8 = pd.DataFrame({
    "sci_sum": [data_HK["sci_sum"].mean()] * 2,
    "rel_sum": [data_HK["rel_sum"].min(), data_HK["rel_sum"].max()],
    "sex": ["female"] * 2,
    "age": [data_HK["age"].mean()] * 2,
    "edu": ["tertiary"] * 2
})
pred8 = model8.predict(new8)  # type="response" → expected counts (lambda)
print(pred8)

##### [MODEL 9:] Negative binomial regression
model9 = smf.negativebinomial("chldrn ~ rel_sum + sci_sum + sex + age + edu",
                              data=data_HK).fit()
print(model9.summary())

# Model 9: Prediction — same newdata as Model 8
pred9 = model9.predict(new8)
print(pred9)

In some cases, we have nested data, meaning that the clustering of the data might have its own effect on the dependent variable. If so, we can apply multilevel models. In Model 10.1, we recall our initial model, the simple linear regression model with religion index as dependent variable, and science index as the predictor.

We switch now to the overall worldwide dataset. In Model 10.2, we estimate the same model, but we account for the clustering of observations on the country level. We add a country-level fixed effect.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

##### [MODEL 10:] Multilevel linear regression model

# Fixed effects only (the lm benchmark)
model10_1 = smf.ols("rel_sum ~ sci_sum", data=data).fit()
print(model10_1.summary())

# Mixed model: random intercepts for country
model10_2 = smf.mixedlm(
    "rel_sum ~ sci_sum",
    data=data,
    groups=data["cntry"]
).fit()
print(model10_2.summary())

# Model 10.2: Prediction for all countries at their mean sci_sum
new10 = pd.DataFrame({
    "sci_sum": [data["sci_sum"].mean()] * data["cntry"].nunique(),
    "cntry": data["cntry"].unique()
})

# Fixed-effects-only prediction (like predict(..., re.form = ~0) in R):
pred_fixed = model10_2.predict(new10)
print(pred_fixed)

# With country random effects (R's default predict.lmer behavior):
random_effects = model10_2.random_effects  # dict: {group: DataFrame with 'Group' col}
re = new10["cntry"].map({k: v["Group"] for k, v in random_effects.items()})
pred_with_re = pred_fixed + re.values
new10["prediction"] = pred_with_re
print(new10)

For the additional longitudional model, please have a look at the R Markdown file.

This is an overview over standard regression techniques in sociology. There are, however, many variations of these techniques, and some of them were not covered yet.
